# 04 — West Virginia FIA: regional context for the Fernow watersheds
**ArcGIS Pro Notebooks or Jupyter** · FIADB state tables for WV (`WV_PLOT`, `WV_COND`, `WV_PLOTGEOM`, `WV_COND_DWM_CALC`).

The Fernow design has one watershed per treatment. FIA cannot fix that, but it can answer three questions a reviewer
will ask of any single-site result:

| § | Question | FIA data |
|---|---|---|
| 1 | **Representativeness** — where do REF, DLC and STS sit in the distribution of WV hardwood stands? | latest-visit conditions |
| 2 | **Generality of the harvest effect** — is the Fernow stocking reduction typical of partial cutting across WV, after terrain/site covariates? | latest visits, OLS + HC3 |
| 3 | **Storage vs. sequestration, regionally** — do high-stock stands still accumulate? how fast do cut stands regrow? | remeasured condition pairs |
| 4 | **Forest floor & dead wood (measured)** — does cutting reduce forest-floor C regionally, as Fernow OH suggests? | DWM plots |
| 5 | Modeled FIA carbon pools — descriptive only, with the reason why | COND `CARBON_*` |
| 6–7 | Optional: export plots to a feature class (arcpy); live-tree carbon if `WV_TREE.csv` is added | |

**Data caveats stated up front.** (i) FIA public coordinates are fuzzed (≤1.6 km) and some are swapped; they are fine
for ecological-section framing, not for site matching. (ii) `WV_TREE.csv` is not in the repository, so live-tree carbon
is not available; live basal area (`BALIVE`, trees ≥2.54 cm) is the stocking measure — it tracks Fernow AGB-C closely
(notebook 02: R² ≈ 0.95 at plot level). (iii) The `CARBON_LITTER`, `CARBON_SOIL_ORG`, `CARBON_DOWN_DEAD` and
understory columns in `COND` are **model predictions**, not plot measurements (Domke et al. 2016, 2017), so they cannot
provide independent evidence of a management effect. Measured forest-floor and dead-wood C come only from the DWM
subsample (§4). (iv) FIA treatment codes record cutting since the previous visit (~5–7 yr window), so "uncut" means
"no cutting recorded recently", not "never cut."


In [ ]:
# ============================== CONFIG ==============================
import os, sys
NOTEBOOK_DIR = os.getcwd()
FIA_DIR = r'.'                         # folder with WV_*.csv, or the filtered folder from notebook 00 (WV_*.csv.gz)
GDB = os.path.join(r'.', 'gdb_export')  # Fernow tables (for comparison values)
FERNOW_LATLON = (39.06, -79.68)
HARDWOOD = [(500, 599), (800, 899)]     # oak/hickory, maple/beech/birch forest-type groups
MIN_INVYR = 2004                         # annual inventory only (periodic 1989/2000 lack treatment codes)
FERNOW_SECTION = None                    # None = derive from FIA plots within 15 km of Fernow
EXPORT_TO_GDB = False                    # §6: write latest-visit FIA plots to the ArcGIS Pro default geodatabase
# ====================================================================
sys.path.insert(0, NOTEBOOK_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
from fernow_stats import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
QC = []
def qc(step, df):
    QC.append((step, len(df))); print(f'[QC] {step}: {len(df):,} rows')
def fia(name):
    # path to WV_<name>.csv.gz (filtered by notebook 00) or WV_<name>.csv (raw); None if absent
    for ext in ('.csv.gz', '.csv'):
        p = os.path.join(FIA_DIR, f'WV_{name}{ext}')
        if os.path.exists(p): return p
    return None
for t in ['PLOT', 'COND', 'PLOTGEOM', 'COND_DWM_CALC']:
    assert fia(t), f'missing WV_{t}(.csv/.csv.gz) in FIA_DIR'
print('FIA files found:', {t: (os.path.basename(fia(t)) if fia(t) else None) for t in ['PLOT', 'COND', 'PLOTGEOM', 'COND_DWM_CALC', 'TREE', 'SEEDLING']})

## §0 — Load, filter, and log every QC step

In [ ]:
P = pd.read_csv(fia('PLOT'), low_memory=False,
                usecols=['CN', 'INVYR', 'COUNTYCD', 'PLOT', 'PLOT_STATUS_CD', 'MEASYEAR', 'LAT', 'LON', 'ELEV', 'PREV_PLT_CN', 'REMPER'])
C = pd.read_csv(fia('COND'), low_memory=False,
                usecols=['CN', 'PLT_CN', 'INVYR', 'CONDID', 'COND_STATUS_CD', 'CONDPROP_UNADJ', 'FORTYPCD', 'STDAGE', 'STDORGCD',
                         'SITECLCD', 'SLOPE', 'ASPECT', 'OWNGRPCD', 'BALIVE', 'DSTRBCD1', 'DSTRBCD2', 'DSTRBCD3',
                         'TRTCD1', 'TRTCD2', 'TRTCD3', 'TRTYR1', 'TRTYR2', 'TRTYR3',
                         'CARBON_DOWN_DEAD', 'CARBON_LITTER', 'CARBON_SOIL_ORG', 'CARBON_UNDERSTORY_AG', 'CARBON_UNDERSTORY_BG'])
G = pd.read_csv(fia('PLOTGEOM'), low_memory=False, usecols=['CN', 'ECOSUBCD'])
G['ECOSUBCD'] = G.ECOSUBCD.astype(str).str.strip(); G['SECTION'] = G.ECOSUBCD.str[:5]
qc('COND all', C)

c = C[C.INVYR >= MIN_INVYR]; qc(f'annual inventory (INVYR >= {MIN_INVYR})', c)
c = c[c.COND_STATUS_CD == 1]; qc('accessible forest land', c)
hw = np.zeros(len(c), bool)
for lo, hi in HARDWOOD: hw |= c.FORTYPCD.between(lo, hi).values
c = c[hw]; qc('hardwood forest types (500s, 800s)', c)
c = c[c.STDORGCD.fillna(0) == 0]; qc('natural origin (not planted)', c)

def prep(df):
    df = df.merge(P.rename(columns={'CN': 'PLT_CN', 'INVYR': 'INVYR_P'}), on='PLT_CN')
    df = df.merge(G.rename(columns={'CN': 'PLT_CN'}), on='PLT_CN', how='left')
    df['pid'] = df.COUNTYCD.astype(str) + '_' + df.PLOT.astype(str)
    df['elev_m'] = df.ELEV * 0.3048
    df['slope_pct'] = df.SLOPE
    df['northness'] = np.where(df.SLOPE > 0, np.cos(np.deg2rad(df.ASPECT)), 0.0)
    df['cut'] = (df[['TRTCD1', 'TRTCD2', 'TRTCD3']] == 10).any(axis=1)
    df['disturbed'] = (df[['DSTRBCD1', 'DSTRBCD2', 'DSTRBCD3']].fillna(0) > 0).any(axis=1)
    trt_yr = df[['TRTYR1', 'TRTYR2', 'TRTYR3']].where(df[['TRTCD1', 'TRTCD2', 'TRTCD3']].values == 10)
    df['yrs_since_cut'] = np.where(df.cut, df.MEASYEAR - trt_yr.max(axis=1), np.nan)
    df['ftgroup'] = np.where(df.FORTYPCD < 600, 'oak/hickory', 'maple/beech/birch')
    return df

c_multi = prep(c.copy())   # all hardwood conditions incl. partial ones — used for per-acre DWM carbon (§4)
print(f'   BALIVE max before single-condition filter: {c.BALIVE.max():.0f} ft2/ac (artifact of tiny partial conditions)')
c = c[c.CONDPROP_UNADJ == 1]; qc('single-condition plots (CONDPROP_UNADJ = 1) — for BALIVE analyses', c)
print(f'   BALIVE max after: {c.BALIVE.max():.0f} ft2/ac')
c = prep(c)

c_multi['in_section'] = False
if FERNOW_SECTION is None:
    d_km = np.hypot((c.LAT - FERNOW_LATLON[0]) * 111.0, (c.LON - FERNOW_LATLON[1]) * 86.0)
    FERNOW_SECTION = c.loc[d_km < 15, 'SECTION'].mode().iat[0]
    print(f'Fernow ecological section (mode of FIA plots within 15 km): {FERNOW_SECTION}; subsections nearby:',
          c.loc[d_km < 15, 'ECOSUBCD'].value_counts().head(3).to_dict())
c['in_section'] = c.SECTION == FERNOW_SECTION; c_multi['in_section'] = c_multi.SECTION == FERNOW_SECTION

latest = c.sort_values('INVYR').groupby('pid').tail(1).copy(); qc('latest visit per plot (cross-sectional sample)', latest)
print(f"Latest-visit: {latest.cut.sum()} with cutting recorded; {latest.in_section.sum()} in section {FERNOW_SECTION}")

In [ ]:
# Fernow comparison values (2022 plots). FIA BALIVE includes live trees >= 1.0 in; Fernow trees also >= 1.0 in.
ft = pd.read_csv(os.path.join(GDB, 'trees.csv')); ft = ft[ft.dbh_cm.notna()]
PLOT_AC = 0.1
fplot = (ft.groupby(['watershed', 'plot_id']).ba_m2.sum() * 10.7639 / PLOT_AC).rename('BA_ft2ac').reset_index()
ter = pd.read_csv(os.path.join(GDB, 'plot_terrain.csv'))[['plot_id', 'dem5_fp', 'slope5_fp']]
fplot = fplot.merge(ter, on='plot_id')
fplot['slope_pct'] = np.tan(np.deg2rad(fplot.slope5_fp)) * 100
FER = fplot.groupby('watershed')[['BA_ft2ac', 'dem5_fp', 'slope_pct']].mean().reindex(ORDER)
FER.index = [LABELS[w] for w in FER.index]
FER.round(1)

## §1 — Representativeness: percentile ranks of the Fernow watersheds
REF is ranked against uncut conditions (and against uncut conditions with stand age ≥ 80 yr, the closest analogue of a
~110-yr unmanaged stand); DLC and STS against conditions with cutting recorded. Both statewide and within Fernow's
ecological section.


In [ ]:
def pct(x, dist): return stats.percentileofscore(dist.dropna(), x, kind='mean')
uncut = latest[~latest.cut]; cutd = latest[latest.cut]
old = uncut[uncut.STDAGE >= 80]
rows = []
for lab, frame in [('WV', latest), (FERNOW_SECTION, latest[latest.in_section])]:
    u, k, o = frame[~frame.cut], frame[frame.cut], frame[(~frame.cut) & (frame.STDAGE >= 80)]
    rows.append(dict(frame=lab, n_uncut=len(u), n_uncut_age80=len(o), n_cut=len(k),
                     REF_pct_vs_uncut=pct(FER.loc['REF', 'BA_ft2ac'], u.BALIVE),
                     REF_pct_vs_uncut_age80=pct(FER.loc['REF', 'BA_ft2ac'], o.BALIVE),
                     DLC_pct_vs_cut=pct(FER.loc['DLC', 'BA_ft2ac'], k.BALIVE),
                     STS_pct_vs_cut=pct(FER.loc['STS', 'BA_ft2ac'], k.BALIVE),
                     uncut_median_BA=u.BALIVE.median(), cut_median_BA=k.BALIVE.median(),
                     REF_elev_pct=pct(FER.loc['REF', 'dem5_fp'], frame.elev_m), DLC_elev_pct=pct(FER.loc['DLC', 'dem5_fp'], frame.elev_m)))
rep = pd.DataFrame(rows); rep.round(1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
bins = np.arange(0, 300, 10)
ax[0].hist(uncut.BALIVE, bins, color='0.75', label=f'WV uncut (n={len(uncut)})', density=True)
ax[0].hist(old.BALIVE, bins, histtype='step', color='k', lw=1.5, label=f'uncut, age ≥80 (n={len(old)})', density=True)
ax[0].axvline(FER.loc['REF', 'BA_ft2ac'], color=COLORS['WS4'], lw=2.5, label='Fernow REF')
ax[1].hist(cutd.BALIVE, bins, color='0.75', label=f'WV cut, recorded (n={len(cutd)})', density=True)
for w in ['WS2', 'WS5']:
    ax[1].axvline(FER.loc[LABELS[w], 'BA_ft2ac'], color=COLORS[w], lw=2.5, label=f'Fernow {LABELS[w]}')
for a in ax: a.set_xlabel('Live basal area (ft$^2$ ac$^{-1}$)'); a.legend(fontsize=8)
ax[0].set_title('Unmanaged reference vs. WV uncut hardwoods'); ax[1].set_title('Managed watersheds vs. recently cut WV hardwoods')
plt.tight_layout(); plt.show()

**Reading §1.** A high REF percentile means the Fernow reference is an unusually well-stocked stand for West Virginia —
so the REF − managed gap at Fernow is likely **larger** than the average contrast between cut and uncut stands statewide.
Say so in the Discussion rather than letting readers generalize the 2× AGB-C ratio.


## §2 — Is the harvest effect general? Covariate-adjusted cut vs. uncut
`BALIVE ~ cut + elevation + slope + northness + site class + forest-type group (+ section)`, OLS with HC3 standard errors,
one condition per plot (latest visit). Stand age is **not** a covariate — it is itself changed by cutting (a
post-treatment variable), so conditioning on it would absorb part of the effect. The Fernow effect for comparison is
the plot-based BA difference (managed − REF) as a percentage of REF.


In [ ]:
def design(df):
    X = [df.cut.astype(float).values, df.elev_m.values, df.slope_pct.values, df.northness.values,
         (df.ftgroup == 'maple/beech/birch').astype(float).values]
    names = ['cut', 'elev_m', 'slope_pct', 'northness', 'maple_beech_birch']
    for s in sorted(df.SITECLCD.dropna().unique())[1:]:
        X.append((df.SITECLCD == s).astype(float).values); names.append(f'siteclass_{int(s)}')
    return X, names
out = []
for lab, frame in [('WV', latest), (FERNOW_SECTION, latest[latest.in_section])]:
    f = frame.dropna(subset=['BALIVE', 'elev_m', 'slope_pct', 'SITECLCD'])
    X, names = design(f); m = ols(f.BALIVE.values, X, names)
    b = m['table'].loc['cut']; base = f.loc[~f.cut, 'BALIVE'].mean()
    out.append(dict(frame=lab, n=len(f), n_cut=int(f.cut.sum()), cut_effect_ft2ac=b.coef,
                    CI_lo=b.coef - 1.96 * b.SE_HC3, CI_hi=b.coef + 1.96 * b.SE_HC3, p_HC3=b.p_HC3,
                    pct_of_uncut_mean=b.coef / base * 100, R2=m['r2']))
reg = pd.DataFrame(out)
fern = (FER.loc[['DLC', 'STS'], 'BA_ft2ac'] / FER.loc['REF', 'BA_ft2ac'] - 1) * 100
print('Fernow plot-based BA difference vs REF (%):', fern.round(0).to_dict())
reg.round(2)

In [ ]:
# Effect by years since cutting (0-6 yr window available in FIA)
k = latest[latest.cut].copy()
k['ysc_bin'] = pd.cut(k.yrs_since_cut, [-0.1, 2, 4, 7], labels=['0-2', '3-4', '5-7'])
print(k.groupby('ysc_bin').BALIVE.agg(['count', 'median', 'mean']).round(1))
print(f'Uncut median BALIVE: {uncut.BALIVE.median():.1f}')

## §3 — Storage vs. sequestration across West Virginia (remeasured conditions)
Each condition is linked to its previous visit (`PREV_PLT_CN`, same `CONDID`), both visits single-condition forest.
Annual change in live basal area = ΔBALIVE / REMPER. Three groups:
- **undisturbed** — no cutting or disturbance recorded at either visit;
- **recovering** — cutting recorded at the previous visit, none since (analogue of Fernow's between-entry regrowth);
- **cut in interval** — cutting recorded at the current visit.

For inference, one pair per plot (the most recent). The Fernow census intervals from notebook 02
(`fernow_census_periods.csv`) are overlaid in the same units.


In [ ]:
prev = C[['PLT_CN', 'CONDID', 'BALIVE', 'CONDPROP_UNADJ', 'COND_STATUS_CD', 'TRTCD1', 'TRTCD2', 'TRTCD3',
          'DSTRBCD1', 'DSTRBCD2', 'DSTRBCD3']].copy()
prev['cut0'] = (prev[['TRTCD1', 'TRTCD2', 'TRTCD3']] == 10).any(axis=1)
prev['dist0'] = (prev[['DSTRBCD1', 'DSTRBCD2', 'DSTRBCD3']].fillna(0) > 0).any(axis=1)
prev = prev.rename(columns={'PLT_CN': 'PREV_PLT_CN', 'BALIVE': 'BA0', 'CONDPROP_UNADJ': 'CP0', 'COND_STATUS_CD': 'CS0'})
pairs = c.merge(prev[['PREV_PLT_CN', 'CONDID', 'BA0', 'CP0', 'CS0', 'cut0', 'dist0']], on=['PREV_PLT_CN', 'CONDID'])
pairs = pairs[(pairs.CP0 == 1) & (pairs.CS0 == 1) & (pairs.REMPER > 0)]; qc('remeasured single-condition pairs', pairs)
pairs['dBA'] = (pairs.BALIVE - pairs.BA0) / pairs.REMPER
pairs['group'] = np.select([pairs.cut, pairs.cut0 & ~pairs.cut, ~pairs.cut & ~pairs.cut0 & ~pairs.disturbed & ~pairs.dist0],
                           ['cut in interval', 'recovering', 'undisturbed'], 'other disturbance')
lp = pairs.sort_values('INVYR').groupby('pid').tail(1); qc('one pair per plot (latest)', lp)
print(lp.groupby('group').dBA.describe()[['count', '50%', 'mean', 'std']].round(2))

In [ ]:
und = lp[lp.group == 'undisturbed'].copy()
und['bin'] = pd.cut(und.BA0, [0, 60, 90, 120, 150, 180, 300])
tab = und.groupby('bin').dBA.agg(['count', 'median']).assign(
    CI=lambda t: [np.round(boot_ci(und.loc[und.bin == b, 'dBA'].values), 2) for b in t.index])
print('Undisturbed stands: annual change in live BA by starting BA (ft2/ac/yr)'); print(tab)
m = ols(und.dBA.values, [und.BA0.values], ['BA0'])
b0, b1 = m['table'].coef
print(f'\nOLS dBA = {b0:.2f} + {b1:.4f} x BA0 (HC3 p = {m["table"].loc["BA0", "p_HC3"]:.2e}, R2 = {m["r2"]:.3f}); '
      f'BA at zero net change (extrapolated) ~ {-b0 / b1:.0f} ft2/ac')
hi = und[und.BA0 >= 180].dBA
print(f'Stands starting >=180 ft2/ac (REF-like): median {hi.median():.2f} ft2/ac/yr, 95% CI {np.round(boot_ci(hi.values), 2)}, '
      f'{(hi > 0).mean() * 100:.0f}% gained BA, n = {len(hi)}')
rec = lp[lp.group == 'recovering'].dBA; comp = und[und.BA0.between(40, 110)].dBA
print(f'Recovering stands: median {rec.median():.2f} (n = {len(rec)}); undisturbed stands at similar starting BA (40-110): '
      f'median {comp.median():.2f}; Mann-Whitney p = {stats.mannwhitneyu(rec, comp).pvalue:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(und.BA0, und.dBA, s=6, color='0.7', alpha=0.5, label='WV FIA undisturbed (one pair/plot)')
rp = lp[lp.group == 'recovering']; ax.scatter(rp.BA0, rp.dBA, s=10, color='#bc4749', alpha=0.6, label='WV FIA recovering from cut')
med = und.groupby('bin').agg(x=('BA0', 'median'), y=('dBA', 'median'))
ax.plot(med.x, med.y, 'k-o', lw=2, label='FIA binned median')
pth = os.path.join(NOTEBOOK_DIR, 'fernow_census_periods.csv')
if os.path.exists(pth):
    fp = pd.read_csv(pth)
    for w in ORDER:
        d = fp[fp.watershed == w]; ax.scatter(d.BA0_ft2ac, d.rate_BA_ft2ac, s=60, marker='D', color=COLORS[w], edgecolor='k', label=f'Fernow census {LABELS[w]}')
else:
    print('Run notebook 02 first to overlay the Fernow census intervals.')
ax.axhline(0, color='k', lw=0.6); ax.set_ylim(-6, 8)
ax.set_xlabel('Live basal area at start of interval (ft$^2$ ac$^{-1}$)'); ax.set_ylabel('Net change (ft$^2$ ac$^{-1}$ yr$^{-1}$)')
ax.set_title('Accumulation slows with stocking but stays positive'); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

**Reading §3.** This is the regional version of the storage/sequestration argument. If accumulation declines with
stocking but remains positive in the highest-stock stands, then REF is *both* the larger store *and* still a sink —
consistent with continued accumulation in old stands (Luyssaert et al. 2008; Stephenson et al. 2014). If recovering
stands regrow faster than undisturbed stands at similar stocking, the managed watersheds' higher net growth (notebook 02)
is the expected regional response, not a Fernow peculiarity.


## §4 — Forest floor and dead wood, measured (DWM subsample)
DWM is measured on a small subsample of plots and visits. Because DWM carbon is per acre of *condition*, partial
conditions are retained here (the single-condition filter is only needed for `BALIVE`). `WV_COND_DWM_CALC` repeats each
condition once per evaluation (EVALID), so it is de-duplicated first. Carbon columns are in lb ac⁻¹ (FIADB User Guide) → × 0.00112085 = Mg ha⁻¹. Forest floor = duff +
litter carbon; dead wood = coarse (CWD) + fine woody debris. Nonparametric comparison (Mann–Whitney, median difference
with bootstrap 95% CI, Cliff's δ) plus a covariate-adjusted OLS on log(1 + C). One condition per plot.


In [ ]:
D = pd.read_csv(fia('COND_DWM_CALC'), low_memory=False,
                usecols=['CND_CN', 'DUFF_CARBON', 'LITTER_CARBON', 'CWD_CARBON_COND', 'FWD_SM_CARBON_COND',
                         'FWD_MD_CARBON_COND', 'FWD_LG_CARBON_COND'])
D = D.drop_duplicates('CND_CN'); qc('DWM_CALC unique conditions (table repeats each condition per EVALID)', D)
dw = c_multi.merge(D, left_on='CN', right_on='CND_CN'); qc('DWM hardwood conditions, all annual visits', dw)
# one condition per plot; if a plot has a cut condition, keep that one (otherwise the latest) so rare cut records are not discarded
dw = dw.sort_values(['cut', 'INVYR']).groupby('pid').tail(1); qc('DWM one condition per plot', dw)
print(f'   cut conditions with DWM: {int(dw.cut.sum())} — inference is limited; treat §4 as supporting evidence')
dw['forest_floor_C'] = (dw.DUFF_CARBON.fillna(0) + dw.LITTER_CARBON.fillna(0)) * LBAC_TO_MGHA
dw['duff_C'] = dw.DUFF_CARBON * LBAC_TO_MGHA; dw['litter_C'] = dw.LITTER_CARBON * LBAC_TO_MGHA
dw['CWD_C'] = dw.CWD_CARBON_COND * LBAC_TO_MGHA
dw['FWD_C'] = dw[['FWD_SM_CARBON_COND', 'FWD_MD_CARBON_COND', 'FWD_LG_CARBON_COND']].sum(axis=1, min_count=1) * LBAC_TO_MGHA
rows = []
for v in ['forest_floor_C', 'duff_C', 'litter_C', 'CWD_C', 'FWD_C']:
    a = dw.loc[dw.cut, v].dropna(); b = dw.loc[~dw.cut, v].dropna()
    lo, hi = boot_diff_ci(a.values, b.values)
    f = dw.dropna(subset=[v, 'elev_m', 'slope_pct', 'SITECLCD']); X, names = design(f)
    mm = ols(np.log1p(f[v].values), X, names)['table'].loc['cut']
    rows.append(dict(pool=v, n_cut=len(a), n_uncut=len(b), median_cut=a.median(), median_uncut=b.median(),
                     median_diff=a.median() - b.median(), CI_lo=lo, CI_hi=hi, MW_p=stats.mannwhitneyu(a, b).pvalue,
                     cliffs_delta=cliffs_delta(a.values, b.values), adj_pct_change=(np.exp(mm.coef) - 1) * 100, adj_p_HC3=mm.p_HC3))
dwm_res = pd.DataFrame(rows); dwm_res['MW_p_FDR'] = bh_fdr(dwm_res.MW_p); dwm_res.round(3)

**Link to Fernow.** If recently cut FIA conditions show lower forest-floor C, that supports the direction of the Fernow
OH trend independently of the OH mass problem (notebook 03, QC-2). If no difference is detected, the Fernow OH trend has
no regional support and, combined with QC-2, should not be a headline claim. A CWD increase after cutting is a C pool the
2022 Fernow sampling did not measure — report it as an omitted pool that partly offsets the aboveground difference in the
short term (it is also the residue source invoked for the subsoil light-fraction mechanism).


## §5 — Modeled FIA carbon pools (descriptive only)
Short tons ac⁻¹ × 2.2417 = Mg ha⁻¹. Soil organic C is the Domke et al. (2017) model prediction to 1 m; litter is the
Domke et al. (2016) model. These are predicted from site, climate and stand attributes, so a cut/uncut difference here
reflects model inputs, not an observed soil response. Shown so the manuscript can place Fernow's *measured* 0–40 cm
mineral C next to the regional *modeled* 0–100 cm value without implying they are comparable measurements.


In [ ]:
mod = latest[['cut', 'in_section', 'CARBON_LITTER', 'CARBON_SOIL_ORG', 'CARBON_DOWN_DEAD', 'CARBON_UNDERSTORY_AG']].copy()
for col in ['CARBON_LITTER', 'CARBON_SOIL_ORG', 'CARBON_DOWN_DEAD', 'CARBON_UNDERSTORY_AG']:
    mod[col] = mod[col] * TONAC_TO_MGHA
print(f'Modeled pools (Mg C/ha), latest visit, section {FERNOW_SECTION}:')
print(mod[mod.in_section].groupby('cut').median().drop(columns='in_section').round(1))
print('\nFernow measured mineral soil C 0-40 cm (notebook 01): REF 83.7, DLC 80.4, STS 60.7 Mg C/ha (different depth and method).')

## §6 — Optional: FIA plots to a feature class (ArcGIS Pro only)
Writes the latest-visit filtered conditions as points (WGS84, fuzzed coordinates) to the project's default geodatabase,
so the section framing and percentile groups can be mapped against the Fernow boundary. Set `EXPORT_TO_GDB = True`.


In [ ]:
if EXPORT_TO_GDB:
    try:
        import arcpy
        tbl = os.path.join(NOTEBOOK_DIR, 'fia_latest_hardwood.csv')
        latest[['pid', 'LAT', 'LON', 'INVYR', 'BALIVE', 'cut', 'STDAGE', 'elev_m', 'SECTION', 'ECOSUBCD']].to_csv(tbl, index=False)
        out_fc = os.path.join(arcpy.env.workspace or arcpy.mp.ArcGISProject('CURRENT').defaultGeodatabase, 'FIA_WV_latest_hardwood')
        arcpy.management.XYTableToPoint(tbl, out_fc, 'LON', 'LAT', coordinate_system=arcpy.SpatialReference(4326))
        print('Wrote', out_fc)
    except Exception as e:
        print('arcpy export skipped:', e)
else:
    print('EXPORT_TO_GDB is False — skipped.')

## §7 — Live-tree carbon (requires `fia_cond_live.csv.gz` from notebook 00)
Repeats §1–§2 in carbon units and adds the **large-tree share** of live AGC (trees ≥50 cm DBH), the structural signature
behind the Fernow result that most REF biomass sits in ≥50 cm stems. FIA carbon comes from FIA's national biomass system,
Fernow's from Jenkins et al. (2003), so percentile ranks are the robust comparison; absolute values carry a method offset.


In [ ]:
lp_path = next((p for p in [os.path.join(FIA_DIR, 'fia_cond_live.csv.gz'), os.path.join(FIA_DIR, 'fia_cond_live.csv')] if os.path.exists(p)), None)
if lp_path is None:
    print('fia_cond_live not found — run notebook 00 on the full FIA download and point FIA_DIR at its output. §7 skipped.')
else:
    live = pd.read_csv(lp_path)
    L = latest.merge(live, on=['PLT_CN', 'CONDID'], how='left')
    L[['live_AGC_MgHa', 'large_AGC_MgHa', 'large_tree_share']] = L[['live_AGC_MgHa', 'large_AGC_MgHa', 'large_tree_share']].fillna(0)
    print(f'Latest-visit hardwood conditions with live-C summary: {L.live_AGC_MgHa.gt(0).sum():,} of {len(L):,}')
    # Fernow 2022 plot values, all stems >= 1 in, published Jenkins (same as notebook 02)
    ft['agb'] = [jenkins_agb_kg(d, g) for d, g in zip(ft.dbh_cm, ft.jenkins_group)]
    fC = (ft.groupby(['watershed', 'plot_id']).agb.sum() * C_FRACTION / 1000 / (PLOT_AC * AC_TO_HA)).rename('AGC').reset_index()
    fL = (ft[ft.dbh_cm >= 50].groupby(['watershed', 'plot_id']).agb.sum() * C_FRACTION / 1000 / (PLOT_AC * AC_TO_HA)).rename('AGC50')
    fC = fC.merge(fL.reset_index(), how='left').fillna({'AGC50': 0})
    FC = fC.groupby('watershed').agg(AGC=('AGC', 'mean'), AGC50=('AGC50', 'mean')).reindex(ORDER)
    FC['large_share'] = FC.AGC50 / FC.AGC; FC.index = [LABELS[w] for w in FC.index]
    print('Fernow 2022 (Jenkins):'); print(FC.round(2))

    rows = []
    for lab, fr in [('WV', L), (FERNOW_SECTION, L[L.in_section])]:
        u, k = fr[~fr.cut], fr[fr.cut]
        rows.append(dict(frame=lab, uncut_median_AGC=u.live_AGC_MgHa.median(), cut_median_AGC=k.live_AGC_MgHa.median(),
                         REF_pct_uncut=pct(FC.loc['REF', 'AGC'], u.live_AGC_MgHa),
                         DLC_pct_cut=pct(FC.loc['DLC', 'AGC'], k.live_AGC_MgHa), STS_pct_cut=pct(FC.loc['STS', 'AGC'], k.live_AGC_MgHa),
                         uncut_median_large_share=u.large_tree_share.median(), cut_median_large_share=k.large_tree_share.median(),
                         REF_large_share_pct_uncut=pct(FC.loc['REF', 'large_share'], u.large_tree_share)))
    print(pd.DataFrame(rows).round(2).to_string(index=False))

    for resp, lab in [('live_AGC_MgHa', 'live AGC (Mg/ha)'), ('large_tree_share', 'large-tree share of AGC')]:
        for flab, fr in [('WV', L), (FERNOW_SECTION, L[L.in_section])]:
            f = fr.dropna(subset=[resp, 'elev_m', 'slope_pct', 'SITECLCD']); X, names = design(f)
            m = ols(f[resp].values, X, names); b = m['table'].loc['cut']
            print(f'{flab:6s} {lab:26s} cut effect {b.coef:+.2f} (95% CI {b.coef - 1.96 * b.SE_HC3:+.2f} to {b.coef + 1.96 * b.SE_HC3:+.2f}), '
                  f'HC3 p = {fmt_p(b.p_HC3)}; {b.coef / f.loc[~f.cut, resp].mean() * 100:+.0f}% of uncut mean')
    fern_pct = (FC.loc[['DLC', 'STS'], 'AGC'] / FC.loc['REF', 'AGC'] - 1) * 100
    print('Fernow AGC difference vs REF (%):', fern_pct.round(0).to_dict())

In [ ]:
pd.DataFrame(QC, columns=['QC step', 'rows'])

---
### References
Bechtold & Patterson (2005) USDA FS GTR SRS-80 (FIA design) · Domke et al. (2016) *Sci. Total Environ.* 557–558:469–478
(litter C model) · Domke et al. (2017) *Ecol. Appl.* 27:1223–1235 (soil C model) · Luyssaert et al. (2008) *Nature*
455:213–215 · Stephenson et al. (2014) *Nature* 507:90–93 · Woodall & Monleon (2008) USDA FS GTR NRS-22 (DWM protocol).
FIADB User Guide (current version) for column definitions and units — confirm units for every column cited in the paper.
